In [0]:
# 4. Dim_Date -- > Create from all date columns.
# | Column              |
# | ------------------- |
# | date_key (YYYYMMDD) |
# | full_date           |
# | day                 |
# | month               |
# | month_name          |
# | quarter             |
# | year                |
# | week_no             |
# | day_name            |

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

date_df = spark.sql("""
SELECT explode(
    sequence(
        to_date('2020-01-01'),
        to_date('2035-12-31'),
        interval 1 day
    )
) AS full_date
""")

dim_date = (
    date_df
    .withColumn("date_key",
                date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("day", dayofmonth("full_date"))
    .withColumn("month", month("full_date"))
    .withColumn("month_name", date_format("full_date", "MMMM"))
    .withColumn("quarter", concat(lit("Q"), quarter("full_date")))
    .withColumn("year", year("full_date"))
    .withColumn("week_no", weekofyear("full_date"))
    .withColumn("day_name", date_format("full_date", "EEEE"))
    .withColumn(
        "is_weekend",
        when(dayofweek("full_date").isin(1,7), "Y").otherwise("N")
    )
)

display(dim_date)

#### cataloge 

In [0]:
dim_date.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.dim_date")

In [0]:
# dim_facilities.write\
#     .format("delta")\
#     .option("mergeSchema","true")\
#     .option("overwriteSchema","true")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .saveAsTable(f"regis_healthcare.gold.sb_dim_facilities")
# print(dim_facilities.count())

In [0]:
# from delta.tables import DeltaTable
# from pyspark.sql import functions as F

# # Load Delta table with correct fully-qualified name
# delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.dim_facilities")
# # Create DataFrame from source table with correct fully-qualified name
# # sb_dim_products
# df_child_products = (
#     spark.table("regis_healthcare.gold.sb_dim_facilities")
#     .select("*")
# )
# # Perform merge
# delta_table.alias("target").merge(
#     source=df_child_products.alias("source"),
#     condition="target.facility_key = source.facility_key"
# ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
# dim_df = spark.sql(f"select * from regis_healthcare.gold.dim_facilities;")
# print(dim_df.count())

# sb_dim_df = spark.sql(f"select * from regis_healthcare.gold.sb_dim_facilities;")
# print(sb_dim_df.count())

#### s3 loading

In [0]:
# gold load to s3
dim_date.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.save(f"s3://regis-healthcare/gold-delta-table/dim_date")

In [0]:
# from delta.tables import DeltaTable

# # ✅ Path to your Delta table stored in S3
# delta_table_path = f"s3://regis-healthcare/gold-delta-table/dim_facilities"

# # ✅ Load target Delta table
# delta_table = DeltaTable.forPath(spark, delta_table_path)

# # ✅ Source DataFrame (example: df_child_products)
# source_df = dim_facilities

# # ✅ Perform MERGE with upsert logic
# (
#     delta_table.alias("target")
#     .merge(
#         source_df.alias("source"),
#         "target.facility_key = source.facility_key"
#     )
#     .whenMatchedUpdateAll()      # Update all columns when matched
#     .whenNotMatchedInsertAll()   # Insert all columns when not matched
#     .execute()
# )
